# Fine-Tuning di Llama 3.1 8B con LoRA (PEFT)
Questo notebook esegue il fine-tuning controllato (SFT) di un Large Language Model utilizzando la tecnica **LoRA (Low-Rank Adaptation)** al fine di realizzare **Information Extraction**. Nello specifico abbiamo due sotto-task simultanei: **Named Entity Recognition (NER)** per identificare i termini e **Relation Extraction (Estrazione di Relazioni)** per capire come sono legati tra loro, ovvero per l'estrazione di triple (es. Vitello Tonnato | USA_INGREDIENTE | Tonno sott'olio)

In [1]:
!pip install -U bitsandbytes transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 102.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.2 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fo

In [2]:
# ===================================
# 1. IPERPARAMETRI E CONFIGURAZIONI 
# ===================================
import os
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,  DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
N_EPOCHS = 3
LR = 2e-4  # Adattato per Causal LM rispetto a BERT
MAX_LEN = 512

# Usiamo una versione pre-quantizzata a 4-bit per far girare Llama 3.1 8B gratis sulla T4 di Kaggle
MODEL_CHECKPOINT = "unsloth/meta-llama-3.1-8b-bnb-4bit" 
DATASET_FILE = "/kaggle/input/datasets/paolaapap/dataset-triple-culinarie/dataset_triple_culinarie.jsonl"

gc.collect()
torch.cuda.empty_cache()

print("📥 Caricamento del dataset generato dal ChromaDB...")
dataset = load_dataset("json", data_files=DATASET_FILE)["train"]

# Dividiamo in Train e Validation set 
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
val_dataset = dataset_split["test"]

print(f"📊 Dataset pronto. Train: {len(train_dataset)} esempi, Validation: {len(val_dataset)} esempi.")

📥 Caricamento del dataset generato dal ChromaDB...


Generating train split: 0 examples [00:00, ? examples/s]

📊 Dataset pronto. Train: 315 esempi, Validation: 35 esempi.


In [3]:
# =========================================================================
# 2. TOKENIZZAZIONE E FORMATTAZIONE PROMPT
# =========================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
tokenizer.pad_token = tokenizer.eos_token  # Gestione del token di fine stringa come padding
tokenizer.padding_side = "right"

def format_and_tokenize_function(examples):
    """Concatena istruzione, input e output nel formato che il modello deve imparare."""
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        # Creiamo la struttura del testo (SFT - Supervised Fine-Tuning)
        text = f"### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Output:\n{out}{tokenizer.eos_token}"
        texts.append(text)
        
    # Tokenizziamo troncando alla lunghezza massima per la memoria della GPU
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding=False)
    
    # Nelle architetture Causal LM, le label per calcolare la Loss corrispondono agli input_ids stessi
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("🧼 Esecuzione della Tokenizzazione dei dati...")
tokenized_train = train_dataset.map(format_and_tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(format_and_tokenize_function, batched=True, remove_columns=val_dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

🧼 Esecuzione della Tokenizzazione dei dati...


Map:   0%|          | 0/315 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

In [4]:
# =========================================================================
# 3. LORA CONFIGURATION (Speculare alla Cella 50 del Professore)
# =========================================================================
lora_config = LoraConfig(
    r=8,                                # Il Rango (Rank) delle matrici sottili LoRA
    lora_alpha=16,                      # Fattore di scala costante matematico
    target_modules=["q_proj", "v_proj"],# Bersagliamo Query e Value di Llama
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM        # Impostiamo Causal LM invece di classificazione sequenze
)

print("🤖 Caricamento del Modello Base e inizializzazione dell'adattatore PEFT...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_CHECKPOINT, 
    torch_dtype=torch.float16,
    device_map="auto"
)

# Applichiamo i parametri LoRA sul modello originale congelato (Cella 51 del Prof)
base_model = prepare_model_for_kbit_training(base_model)
base_model.config.use_cache = False

model_lora = get_peft_model(base_model, lora_config)
model_lora.print_trainable_parameters()

🤖 Caricamento del Modello Base e inizializzazione dell'adattatore PEFT...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


In [ ]:
# =========================================================================
# 4. TRAINING ARGUMENTS E TRAINER (Pulito e Aggiornato)
# =========================================================================
training_args = TrainingArguments(
    output_dir="./culinary_lora_output",
    learning_rate=LR,
    per_device_train_batch_size=1,       
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,  
    gradient_checkpointing_kwargs={"use_reentrant": False},
    num_train_epochs=N_EPOCHS,
    eval_strategy="epoch",               
    save_strategy="epoch",               
    # 💡 Rimosso logging_dir per eliminare il DeprecationWarning
    logging_strategy="steps",
    logging_steps=10,
    fp16=True, 
    optim="paged_adamw_8bit",
    save_total_limit=1,
    report_to="none"                     
)

# Inizializziamo il modulo Trainer di Hugging Face
trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("🏋️ Avvio del processo di Fine-Tuning LoRA...")
trainer.train()

🏋️ Avvio del processo di Fine-Tuning LoRA...


Epoch,Training Loss,Validation Loss


In [ ]:
# =========================================================================
# 5. SALVATAGGIO DELL'ADATTATORE FINALE
# =========================================================================
OUTPUT_ADAPTER_DIR = "./final_culinary_lora_adapter"
model_lora.save_pretrained(OUTPUT_ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)

print(f"\n🎉 Addestramento concluso con successo! I pesi LoRA leggeri sono salvati in: {OUTPUT_ADAPTER_DIR}")
print("Scarica questa cartella per procedere all'integrazione locale.")